# Vision Models Registery

> This module provides a registry for vision models, allowing for easy registration and retrieval of models by name.


In [ ]:
#| default_exp models.vision.registery

In [ ]:
#| hide
from nbdev.showdoc import * 

In [ ]:
#| export
from typing import Dict, Callable, List, Tuple, Optional
import functools


In [ ]:
#| export

_MODEL_REGISTRY: Dict[str, Dict] = {}

def register_model(name: str, variants: List[str]):
    """
    Decorator to automatically register model factories.
    
    Args:
        name: Model architecture name (e.g., 'resnet', 'vit')
        variants: List of supported variants (e.g., ['18', '34', '50'])
    """
    def decorator(factory_fn: Callable):
        _MODEL_REGISTRY[name] = {
            'factory': factory_fn,
            'variants': variants,
        }
        
        @functools.wraps(factory_fn)
        def wrapper(*args, **kwargs):
            return factory_fn(*args, **kwargs)
        
        return wrapper
    return decorator


def list_models() -> Dict[str, List[str]]:
    """
    Return all registered models and their variants.
    
    Returns:
        Dict mapping architecture names to their variants
        Example: {'resnet': ['18', '50'], 'vit': ['small', 'base']}
    """
    return {name: info['variants'] for name, info in _MODEL_REGISTRY.items()}


def list_model_names() -> List[str]:
    """
    Return flat list of full model names.
    
    Returns:
        List of model names in 'architecture_variant' format
        Example: ['resnet_18', 'resnet_50', 'vit_base']
    """
    result = []
    for arch, info in _MODEL_REGISTRY.items():
        for variant in info['variants']:
            result.append(f"{arch}_{variant}")
    return sorted(result)


def parse_model_name(model_name: str) -> Tuple[str, str]:
    """
    Parse 'architecture_variant' into (architecture, variant).
    
    Args:
        model_name: Full model name like 'resnet_50' or 'vit_base'
    
    Returns:
        (architecture, variant) tuple
    
    Raises:
        ValueError: If model_name format is invalid or model not registered
    """
    if '_' not in model_name:
        raise ValueError(
            f"Invalid model_name format: '{model_name}'. "
            f"Expected format: 'architecture_variant' (e.g., 'resnet_50'). "
            f"Available models: {list_model_names()}"
        )
    
    parts = model_name.split('_', 1)  # Split only on first underscore
    architecture, variant = parts[0], parts[1]
    
    # Validate architecture exists
    if architecture not in _MODEL_REGISTRY:
        available_archs = list(_MODEL_REGISTRY.keys())
        raise ValueError(
            f"Unknown architecture: '{architecture}'. "
            f"Available architectures: {available_archs}"
        )
    
    # Validate variant exists for this architecture
    valid_variants = _MODEL_REGISTRY[architecture]['variants']
    if variant not in valid_variants:
        raise ValueError(
            f"Unknown variant '{variant}' for {architecture}. "
            f"Valid variants: {valid_variants}"
        )
    
    return architecture, variant


def get_factory(architecture: str) -> Callable:
    """Get factory function for an architecture."""
    if architecture not in _MODEL_REGISTRY:
        raise ValueError(
            f"Architecture '{architecture}' not registered. "
            f"Available: {list(_MODEL_REGISTRY.keys())}"
        )
    return _MODEL_REGISTRY[architecture]['factory']

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()